# Guía de Laboratorio 2 — Queues y SW Timers en FreeRTOS

**Hardware:** STM32 Nucleo F103RB · 4 LEDs externos · 1 pulsador
**Librerías:** `FreeRTOS.h`, `queue.h`, `timers.h`
**Restricción:** Solo API nativa FreeRTOS + STM32 HAL. Sin CMSIS-RTOS.

```c
/* Pines — configurar en MX_GPIO_Init() vía STM32CubeMX */
#define LED1_PIN   GPIO_PIN_0
#define LED1_PORT  GPIOA
#define LED2_PIN   GPIO_PIN_1
#define LED2_PORT  GPIOA
#define LED3_PIN   GPIO_PIN_4
#define LED3_PORT  GPIOA
#define LED4_PIN   GPIO_PIN_5
#define LED4_PORT  GPIOA
#define BTN_PIN    GPIO_PIN_13
#define BTN_PORT   GPIOC
```

## Desafío 1 — Comunicación por Queues

**Objetivo:** Transferir datos entre tareas de forma segura.

**Comportamiento:**
- **Tarea Productora:** lee el botón, mantiene un contador de clicks y envía el valor (`uint16_t`) a la cola con cada pulsación.
- **Tarea Consumidora:** recibe el valor y lo muestra en los 4 LEDs en **binario** (bit 0 → LED1, … bit 3 → LED4).

### Solución — Desafío 1

```c
#include "FreeRTOS.h"
#include "task.h"
#include "queue.h"
#include "main.h"

static QueueHandle_t xColaClicks = NULL;

/* Tarea Productora: detecta flancos y envía el contador */
static void vTarea_Productora(void *pvParams) {
    uint16_t contador = 0;
    uint8_t  anterior = GPIO_PIN_SET;

    for (;;) {
        uint8_t actual = HAL_GPIO_ReadPin(BTN_PORT, BTN_PIN);
        if (actual == GPIO_PIN_RESET && anterior == GPIO_PIN_SET) {
            contador++;
            xQueueSend(xColaClicks, &contador, 0);   /* No bloquear si la cola está llena */
        }
        anterior = actual;
        vTaskDelay(pdMS_TO_TICKS(20));   /* Debounce de 20 ms */
    }
}

static const uint16_t  led_pin[]  = {LED1_PIN, LED2_PIN, LED3_PIN, LED4_PIN};
static GPIO_TypeDef   *led_port[] = {LED1_PORT, LED2_PORT, LED3_PORT, LED4_PORT};

/* Tarea Consumidora: muestra el valor en binario */
static void vTarea_Consumidora(void *pvParams) {
    uint16_t valor;
    for (;;) {
        xQueueReceive(xColaClicks, &valor, portMAX_DELAY);
        for (int i = 0; i < 4; i++) {
            GPIO_PinState s = (valor & (1u << i)) ? GPIO_PIN_SET : GPIO_PIN_RESET;
            HAL_GPIO_WritePin(led_port[i], led_pin[i], s);
        }
        vTaskDelay(pdMS_TO_TICKS(500));   /* Simula consumidora más lenta que la productora */
    }
}

int main(void) {
    HAL_Init();
    SystemClock_Config();
    MX_GPIO_Init();

    xColaClicks = xQueueCreate(10, sizeof(uint16_t));

    xTaskCreate(vTarea_Productora,  "PROD", 128, NULL, 2, NULL);
    xTaskCreate(vTarea_Consumidora, "CONS", 128, NULL, 1, NULL);

    vTaskStartScheduler();
    for (;;);
}
```

### Análisis — Desafío 1

- **Si la consumidora es más lenta:** la cola acumula elementos. Si se llena, `xQueueSend(..., 0)` retorna `errQUEUE_FULL` y el click se **pierde silenciosamente**.
- **Cola de 1 elemento:** se pierde todo click mientras la consumidora procesa el anterior.
- **Cola más grande:** permite buffear más eventos pero la visualización queda **retrasada** (el display muestra clicks pasados, no el estado actual).

## Desafío 2 — Paso de Estructuras

**Objetivo:** Gestionar múltiples datos en un solo mensaje de cola.

**Comportamiento:** La tarea principal envía estructuras `{led_id, comando}` a una cola. Una tarea de gestión de periféricos las procesa y ejecuta la acción en el LED correspondiente. Se generan múltiples patrones de encendido.

### Solución — Desafío 2

```c
#include "FreeRTOS.h"
#include "task.h"
#include "queue.h"
#include "main.h"

typedef enum { CMD_APAGAR = 0, CMD_ENCENDER = 1 } LedCmd_t;

typedef struct {
    uint8_t  led_id;   /* 0 = LED1 … 3 = LED4 */
    LedCmd_t comando;
} MsgLed_t;

static QueueHandle_t xColaLEDs = NULL;

static const uint16_t  led_pin[]  = {LED1_PIN, LED2_PIN, LED3_PIN, LED4_PIN};
static GPIO_TypeDef   *led_port[] = {LED1_PORT, LED2_PORT, LED3_PORT, LED4_PORT};

/* Tarea principal: genera dos patrones de encendido */
static void vTarea_Principal(void *pvParams) {
    MsgLed_t msg;

    /* Patrón 1: secuencia 1 → 2 → 3 → 4 */
    for (uint8_t i = 0; i < 4; i++) {
        msg = (MsgLed_t){i, CMD_ENCENDER};
        xQueueSend(xColaLEDs, &msg, portMAX_DELAY);
        vTaskDelay(pdMS_TO_TICKS(300));
        msg.comando = CMD_APAGAR;
        xQueueSend(xColaLEDs, &msg, portMAX_DELAY);
    }

    /* Patrón 2: todos ON → todos OFF (× 2) */
    for (int rep = 0; rep < 2; rep++) {
        for (uint8_t i = 0; i < 4; i++) {
            msg = (MsgLed_t){i, CMD_ENCENDER};
            xQueueSend(xColaLEDs, &msg, portMAX_DELAY);
        }
        vTaskDelay(pdMS_TO_TICKS(500));
        for (uint8_t i = 0; i < 4; i++) {
            msg = (MsgLed_t){i, CMD_APAGAR};
            xQueueSend(xColaLEDs, &msg, portMAX_DELAY);
        }
        vTaskDelay(pdMS_TO_TICKS(500));
    }

    vTaskDelete(NULL);
}

/* Tarea gestión de periféricos */
static void vTarea_Perifericos(void *pvParams) {
    MsgLed_t msg;
    for (;;) {
        xQueueReceive(xColaLEDs, &msg, portMAX_DELAY);
        if (msg.led_id < 4) {
            GPIO_PinState s = (msg.comando == CMD_ENCENDER) ? GPIO_PIN_SET : GPIO_PIN_RESET;
            HAL_GPIO_WritePin(led_port[msg.led_id], led_pin[msg.led_id], s);
        }
    }
}

int main(void) {
    HAL_Init();
    SystemClock_Config();
    MX_GPIO_Init();

    xColaLEDs = xQueueCreate(20, sizeof(MsgLed_t));

    xTaskCreate(vTarea_Principal,   "MAIN", 256, NULL, 1, NULL);
    xTaskCreate(vTarea_Perifericos, "PERI", 128, NULL, 2, NULL);

    vTaskStartScheduler();
    for (;;);
}
```

### Análisis — Desafío 2

| Método | Bytes copiados a la cola | Seguridad |
|--------|--------------------------|-----------|
| Por **copia** (`sizeof(MsgLed_t)` = 2 B) | 2 B por mensaje | ✅ Siempre seguro |
| Por **puntero** (`sizeof(MsgLed_t*)` = 4 B en STM32) | 4 B por mensaje | ⚠️ Peligroso si la estructura está en el stack de la tarea productora |

- **Envío por puntero** es más eficiente para estructuras grandes, pero exige que la estructura tenga vida útil mayor que el tiempo de procesamiento (memoria dinámica o estática, nunca el stack local de la tarea productora).
- En sistemas embebidos con estructuras pequeñas (≤ 16 B), el **envío por copia es el patrón recomendado**.

## Desafío 3 — Multi-recepción con Queue Sets

**Objetivo:** Gestionar múltiples fuentes de datos asincrónicas desde una única tarea receptora sin pérdida de eventos.

**Comportamiento:**
- `Cola_Velocidad` → envía un delay en ms (aleatorio 100–300 ms) cada 5 segundos.
- `Cola_Modo` → envía `'D'` o `'I'` al presionar el botón (alterna sentido).
- Tarea **Procesadora** bloquea en el Queue Set y ejecuta una secuencia Knight Rider ajustando velocidad y sentido según los mensajes recibidos.

### Solución — Desafío 3

```c
#include "FreeRTOS.h"
#include "task.h"
#include "queue.h"
#include "semphr.h"
#include "main.h"

static QueueHandle_t    xCola_Vel  = NULL;
static QueueHandle_t    xCola_Modo = NULL;
static QueueSetHandle_t xSet       = NULL;
static SemaphoreHandle_t xSemBtn   = NULL;

static const uint16_t  led_pin[]  = {LED1_PIN, LED2_PIN, LED3_PIN, LED4_PIN};
static GPIO_TypeDef   *led_port[] = {LED1_PORT, LED2_PORT, LED3_PORT, LED4_PORT};

void HAL_GPIO_EXTI_Callback(uint16_t GPIO_Pin) {
    if (GPIO_Pin == BTN_PIN) {
        BaseType_t xHPTW = pdFALSE;
        xSemaphoreGiveFromISR(xSemBtn, &xHPTW);
        portYIELD_FROM_ISR(xHPTW);
    }
}

/* Tarea Velocidad: nuevo valor aleatorio cada 5 s */
static void vTarea_Vel(void *pvParams) {
    for (;;) {
        uint32_t v = 100 + (HAL_GetTick() % 201);   /* 100..300 ms pseudo-aleatorio */
        xQueueSend(xCola_Vel, &v, 0);
        vTaskDelay(pdMS_TO_TICKS(5000));
    }
}

/* Tarea Modo: alterna 'D' / 'I' con cada click */
static void vTarea_Modo(void *pvParams) {
    char modo = 'D';
    for (;;) {
        xSemaphoreTake(xSemBtn, portMAX_DELAY);
        modo = (modo == 'D') ? 'I' : 'D';
        xQueueSend(xCola_Modo, &modo, 0);
        vTaskDelay(pdMS_TO_TICKS(200));   /* Anti-rebote */
    }
}

/* Tarea Procesadora: Knight Rider con Queue Set */
static void vTarea_Procesadora(void *pvParams) {
    uint32_t delay_ms = 200;
    int pos = 0, dir = 1;
    QueueSetMemberHandle_t xActivo;

    for (;;) {
        /* Actualiza LEDs */
        for (int i = 0; i < 4; i++)
            HAL_GPIO_WritePin(led_port[i], led_pin[i], GPIO_PIN_RESET);
        HAL_GPIO_WritePin(led_port[pos], led_pin[pos], GPIO_PIN_SET);

        /* Bloquea hasta que llegue algo o expire el delay del Knight Rider */
        xActivo = xQueueSelectFromSet(xSet, pdMS_TO_TICKS(delay_ms));

        if (xActivo == xCola_Vel) {
            uint32_t nueva;
            xQueueReceive(xCola_Vel, &nueva, 0);
            delay_ms = nueva;
        } else if (xActivo == xCola_Modo) {
            char m;
            xQueueReceive(xCola_Modo, &m, 0);
            dir = (m == 'D') ? 1 : -1;
        } else {
            /* Timeout: avanza al siguiente LED */
            pos += dir;
            if (pos >= 4) { pos = 2; dir = -1; }
            if (pos < 0)  { pos = 1; dir =  1; }
        }
    }
}

int main(void) {
    HAL_Init();
    SystemClock_Config();
    MX_GPIO_Init();

    xSemBtn    = xSemaphoreCreateBinary();
    xCola_Vel  = xQueueCreate(5, sizeof(uint32_t));
    xCola_Modo = xQueueCreate(5, sizeof(char));
    /* Capacidad del set >= suma de capacidades de las colas miembro */
    xSet = xQueueCreateSet(10);
    xQueueAddToSet(xCola_Vel,  xSet);
    xQueueAddToSet(xCola_Modo, xSet);

    xTaskCreate(vTarea_Vel,        "VEL",  128, NULL, 1, NULL);
    xTaskCreate(vTarea_Modo,       "MODO", 128, NULL, 2, NULL);
    xTaskCreate(vTarea_Procesadora,"PROC", 256, NULL, 1, NULL);

    vTaskStartScheduler();
    for (;;);
}
```

### Análisis — Desafío 3

| Criterio | Queue Set | Polling (`xQueueReceive` timeout 0) |
|----------|-----------|--------------------------------------|
| Consumo de CPU en espera | 0% (tarea bloqueada) | 100% en su prioridad |
| Latencia de respuesta | Inmediata al llegar un dato | 0 ms (loop continuo) |
| Complejidad | Moderada (requiere setup del Set) | Simple |
| Starvation de otras tareas | No | Posible si tiene alta prioridad |

- **Tamaño del Queue Set:** debe ser ≥ a la suma de los máximos elementos de todas las colas miembro. Si se supera esa capacidad, `xQueueAddToSet` falla y el sistema queda en estado inconsistente.

## Desafío 4 — Distribución de Datos con Queue Peek

**Objetivo:** Permitir que múltiples tareas lean el mismo dato sin consumirlo, habilitando procesamiento paralelo de un mismo evento.

**Comportamiento:**
- Cola Global de **1 elemento** (`uint16_t`). La Productora usa `xQueueOverwrite`.
- **Tarea A:** `xQueuePeek` → si el valor es par, parpadea LED1.
- **Tarea B:** `xQueuePeek` → si el valor es impar, parpadea LED2.
- **Tarea Limpieza:** `xQueueReceive` después de que A y B hayan procesado el dato.

### Solución — Desafío 4

```c
#include "FreeRTOS.h"
#include "task.h"
#include "queue.h"
#include "semphr.h"
#include "main.h"

static QueueHandle_t     xCola_Global = NULL;
static SemaphoreHandle_t xSemA        = NULL;   /* A señaliza que procesó */
static SemaphoreHandle_t xSemB        = NULL;   /* B señaliza que procesó */

/* Tarea Productora: botón → contador → xQueueOverwrite (nunca bloquea) */
static void vTarea_Productora(void *pvParams) {
    uint16_t cnt = 0;
    uint8_t  ant = GPIO_PIN_SET;
    for (;;) {
        uint8_t act = HAL_GPIO_ReadPin(BTN_PORT, BTN_PIN);
        if (act == GPIO_PIN_RESET && ant == GPIO_PIN_SET) {
            cnt++;
            xQueueOverwrite(xCola_Global, &cnt);   /* Sobrescribe si la cola está llena */
        }
        ant = act;
        vTaskDelay(pdMS_TO_TICKS(20));
    }
}

/* Tarea A: peek → parpadea LED1 si par */
static void vTarea_A(void *pvParams) {
    uint16_t val;
    for (;;) {
        if (xQueuePeek(xCola_Global, &val, portMAX_DELAY) == pdTRUE) {
            if (val % 2 == 0) {
                HAL_GPIO_WritePin(LED1_PORT, LED1_PIN, GPIO_PIN_SET);
                vTaskDelay(pdMS_TO_TICKS(200));
                HAL_GPIO_WritePin(LED1_PORT, LED1_PIN, GPIO_PIN_RESET);
            }
            xSemaphoreGive(xSemA);   /* Señaliza: terminé de procesar */
        }
        vTaskDelay(pdMS_TO_TICKS(50));
    }
}

/* Tarea B: peek → parpadea LED2 si impar */
static void vTarea_B(void *pvParams) {
    uint16_t val;
    for (;;) {
        if (xQueuePeek(xCola_Global, &val, portMAX_DELAY) == pdTRUE) {
            if (val % 2 != 0) {
                HAL_GPIO_WritePin(LED2_PORT, LED2_PIN, GPIO_PIN_SET);
                vTaskDelay(pdMS_TO_TICKS(200));
                HAL_GPIO_WritePin(LED2_PORT, LED2_PIN, GPIO_PIN_RESET);
            }
            xSemaphoreGive(xSemB);
        }
        vTaskDelay(pdMS_TO_TICKS(50));
    }
}

/* Tarea Limpieza: espera que A y B hayan procesado, luego vacía la cola */
static void vTarea_Limpieza(void *pvParams) {
    uint16_t dummy;
    for (;;) {
        xSemaphoreTake(xSemA, portMAX_DELAY);
        xSemaphoreTake(xSemB, portMAX_DELAY);
        xQueueReceive(xCola_Global, &dummy, 0);
    }
}

int main(void) {
    HAL_Init();
    SystemClock_Config();
    MX_GPIO_Init();

    xCola_Global = xQueueCreate(1, sizeof(uint16_t));
    xSemA        = xSemaphoreCreateBinary();
    xSemB        = xSemaphoreCreateBinary();

    xTaskCreate(vTarea_Productora, "PROD",  128, NULL, 3, NULL);
    xTaskCreate(vTarea_A,          "A",     128, NULL, 1, NULL);
    xTaskCreate(vTarea_B,          "B",     128, NULL, 1, NULL);
    xTaskCreate(vTarea_Limpieza,   "CLEAN", 128, NULL, 1, NULL);

    vTaskStartScheduler();
    for (;;);
}
```

### Análisis — Desafío 4

- **Si Tarea A usara `xQueueReceive`:** consumiría el dato de la cola; Tarea B encontraría la cola vacía y `xQueuePeek` bloquearía esperando un nuevo dato → **Tarea B pierde el evento**.
- **Garantía de que ambas leyeron antes del borrado:** los semáforos `xSemA` y `xSemB` actúan como barreras. La Tarea Limpieza solo ejecuta `xQueueReceive` después de tomar ambos semáforos → el dato permanece en la cola hasta que las dos lo procesaron.
- **Limitación:** si la Productora genera un nuevo dato (via `xQueueOverwrite`) antes de que A y B terminen, el dato viejo se sobrescribe. Este diseño asume que la tasa de producción es menor que el tiempo de procesamiento de A y B.

## Desafío 5 — Temporizadores de Seguridad (One-Shot)

**Objetivo:** Implementar un "Watchdog" de software que desactiva procesos tras inactividad.

**Comportamiento:**
- LED1 parpadea constantemente (tarea independiente).
- Al presionar el botón (ISR): enciende LED2 y **arranca o reinicia** un timer One-Shot de 5 s.
- Si se vuelve a presionar antes de los 5 s: `xTimerReset` → el timer se reinicia.
- Al expirar: callback apaga LED2 y enciende LED3 por 1 s (señal de "Timeout").

### Solución — Desafío 5

```c
#include "FreeRTOS.h"
#include "task.h"
#include "timers.h"
#include "semphr.h"
#include "main.h"

static TimerHandle_t     xTimer_WDG  = NULL;   /* One-Shot 5 s */
static TimerHandle_t     xTimer_LED3 = NULL;   /* One-Shot 1 s: apaga LED3 */
static SemaphoreHandle_t xSemBtn     = NULL;

/* Callback auxiliar: apaga LED3 tras 1 s */
static void vCB_LED3_Off(TimerHandle_t xT) {
    HAL_GPIO_WritePin(LED3_PORT, LED3_PIN, GPIO_PIN_RESET);
}

/* Callback del Watchdog: apaga LED2, enciende LED3 por 1 s */
static void vCB_Watchdog(TimerHandle_t xT) {
    HAL_GPIO_WritePin(LED2_PORT, LED2_PIN, GPIO_PIN_RESET);
    HAL_GPIO_WritePin(LED3_PORT, LED3_PIN, GPIO_PIN_SET);
    xTimerStart(xTimer_LED3, 0);   /* Dispara el timer de 1 s para apagar LED3 */
    /*
     * NOTA: NO se puede usar vTaskDelay() aquí.
     * Este callback se ejecuta en el contexto de la tarea Timer Daemon.
     * Bloquear la Timer Daemon detendría todos los demás timers del sistema.
     * Por eso usamos un segundo timer One-Shot para gestionar la espera de 1 s.
     */
}

void HAL_GPIO_EXTI_Callback(uint16_t GPIO_Pin) {
    if (GPIO_Pin == BTN_PIN) {
        BaseType_t xHPTW = pdFALSE;
        xSemaphoreGiveFromISR(xSemBtn, &xHPTW);
        portYIELD_FROM_ISR(xHPTW);
    }
}

/* Tarea LED1: parpadea independientemente del resto */
static void vTarea_LED1(void *pvParams) {
    for (;;) {
        HAL_GPIO_TogglePin(LED1_PORT, LED1_PIN);
        vTaskDelay(pdMS_TO_TICKS(500));
    }
}

/* Tarea Botón: gestiona el timer Watchdog */
static void vTarea_Boton(void *pvParams) {
    for (;;) {
        xSemaphoreTake(xSemBtn, portMAX_DELAY);
        HAL_GPIO_WritePin(LED2_PORT, LED2_PIN, GPIO_PIN_SET);
        if (xTimerIsTimerActive(xTimer_WDG) == pdFALSE)
            xTimerStart(xTimer_WDG, 0);
        else
            xTimerReset(xTimer_WDG, 0);
        vTaskDelay(pdMS_TO_TICKS(200));   /* Anti-rebote */
    }
}

int main(void) {
    HAL_Init();
    SystemClock_Config();
    MX_GPIO_Init();

    xSemBtn      = xSemaphoreCreateBinary();
    xTimer_WDG   = xTimerCreate("WDG", pdMS_TO_TICKS(5000), pdFALSE, NULL, vCB_Watchdog);
    xTimer_LED3  = xTimerCreate("L3",  pdMS_TO_TICKS(1000), pdFALSE, NULL, vCB_LED3_Off);

    xTaskCreate(vTarea_LED1,  "LED1", 128, NULL, 1, NULL);
    xTaskCreate(vTarea_Boton, "BTN",  128, NULL, 2, NULL);

    vTaskStartScheduler();
    for (;;);
}
```

### Análisis — Desafío 5

- **Si se presiona cada 2 s:** `xTimerReset` reinicia el contador a 0 en cada pulsación. El timer nunca expira mientras se siga presionando → LED2 permanece encendido, LED3 nunca se activa.
- **LED1 no se ve afectado:** es una tarea completamente independiente con su propio contexto y `vTaskDelay`. La lógica del botón/timer no interfiere con ella.
- **¿Por qué el callback NO puede usar `vTaskDelay()`?**
  - Los callbacks de Software Timers se ejecutan en el contexto de la **tarea Timer Daemon** (tarea interna de FreeRTOS).
  - Bloquear el Timer Daemon bloquearía **todos** los demás callbacks de timers del sistema.
  - La solución correcta para "esperar N ms en un callback" es usar **otro timer One-Shot**.

## Desafío 6 — Metrónomo de Alerta (Auto-Reload)

**Objetivo:** Gestionar eventos cíclicos sin crear tareas adicionales, optimizando el uso de RAM.

**Comportamiento:**
- Timer Auto-Reload con período inicial 1000 ms → toggle LED4 en cada expiración.
- Cada pulsación del botón reduce el período a la mitad: 1000 → 500 → 250 → 125 → 1000 ms.

### Solución — Desafío 6

```c
#include "FreeRTOS.h"
#include "task.h"
#include "timers.h"
#include "semphr.h"
#include "main.h"

static TimerHandle_t     xTimer_Metro = NULL;
static SemaphoreHandle_t xSemBtn      = NULL;

static const uint32_t periodos_ms[] = {1000, 500, 250, 125};
static uint8_t idx = 0;

/* Callback: toggle LED4. Se ejecuta en el Timer Daemon, no en una tarea propia. */
static void vCB_Metro(TimerHandle_t xT) {
    HAL_GPIO_TogglePin(LED4_PORT, LED4_PIN);
}

void HAL_GPIO_EXTI_Callback(uint16_t GPIO_Pin) {
    if (GPIO_Pin == BTN_PIN) {
        BaseType_t xHPTW = pdFALSE;
        xSemaphoreGiveFromISR(xSemBtn, &xHPTW);
        portYIELD_FROM_ISR(xHPTW);
    }
}

static void vTarea_Boton(void *pvParams) {
    for (;;) {
        xSemaphoreTake(xSemBtn, portMAX_DELAY);
        idx = (idx + 1) % 4;
        /* xTimerChangePeriod reinicia el timer con el nuevo período */
        xTimerChangePeriod(xTimer_Metro, pdMS_TO_TICKS(periodos_ms[idx]), 0);
        vTaskDelay(pdMS_TO_TICKS(200));   /* Anti-rebote */
    }
}

int main(void) {
    HAL_Init();
    SystemClock_Config();
    MX_GPIO_Init();

    xSemBtn      = xSemaphoreCreateBinary();
    xTimer_Metro = xTimerCreate("METRO", pdMS_TO_TICKS(1000), pdTRUE, NULL, vCB_Metro);
    xTimerStart(xTimer_Metro, 0);

    xTaskCreate(vTarea_Boton, "BTN", 128, NULL, 1, NULL);

    vTaskStartScheduler();
    for (;;);
}
```

### Análisis — Desafío 6

| Diseño | RAM de stack | Nota |
|--------|-------------|------|
| Tarea con `vTaskDelay` | ≥ 512 B (128 words × 4 B) | Stack completo por tarea |
| Software Timer | ~40 B (estructura `xTIMER`) | Comparte el stack del Timer Daemon |

- **Ahorro:** ≈ 470 B de RAM por cada toggle cíclico implementado como timer en lugar de tarea.
- **Dónde reside la lógica de toggle:** en el callback `vCB_Metro`, ejecutado por la tarea **Timer Daemon** de FreeRTOS. No existe una tarea de usuario dedicada al LED4.
- **Limitación:** `xTimerChangePeriod` reinicia el contador del timer desde cero al cambiar el período. Esto es deseable aquí, pero hay que tenerlo en cuenta si la fase del ciclo importa.

## Desafío 7 — Sistema de Alarmas Escalonadas (Combinado)

**Objetivo:** Orquestar múltiples temporizadores para crear secuencias complejas de control.

**Comportamiento:**
1. Al presionar el botón: timer Auto-Reload (100 ms) hace parpadear LED1 rápidamente + timer One-Shot de 10 s arranca simultáneamente.
2. Al expirar el One-Shot: detiene el parpadeo de LED1, enciende LED2 fijo (sistema armado).
3. Si se presiona el botón **durante** los 10 s: cancela ambos timers y apaga todos los LEDs.

### Solución — Desafío 7

```c
#include "FreeRTOS.h"
#include "task.h"
#include "timers.h"
#include "semphr.h"
#include "main.h"

static TimerHandle_t     xTimer_Blink = NULL;   /* Auto-Reload 100 ms */
static TimerHandle_t     xTimer_Arm   = NULL;   /* One-Shot 10 s */
static SemaphoreHandle_t xSemBtn      = NULL;
static volatile uint8_t  armando      = 0;      /* Flag de estado */

/* Callback del parpadeo: toggle LED1 */
static void vCB_Blink(TimerHandle_t xT) {
    HAL_GPIO_TogglePin(LED1_PORT, LED1_PIN);
}

/* Callback del One-Shot: proceso completado → sistema armado */
static void vCB_Armado(TimerHandle_t xT) {
    /*
     * Es seguro llamar xTimerStop() desde el callback de otro timer:
     * ambas operaciones encolan comandos en la cola del Timer Daemon,
     * no modifican directamente el estado del timer en ejecución.
     */
    xTimerStop(xTimer_Blink, 0);
    HAL_GPIO_WritePin(LED1_PORT, LED1_PIN, GPIO_PIN_RESET);
    HAL_GPIO_WritePin(LED2_PORT, LED2_PIN, GPIO_PIN_SET);
    armando = 0;
}

void HAL_GPIO_EXTI_Callback(uint16_t GPIO_Pin) {
    if (GPIO_Pin == BTN_PIN) {
        BaseType_t xHPTW = pdFALSE;
        xSemaphoreGiveFromISR(xSemBtn, &xHPTW);
        portYIELD_FROM_ISR(xHPTW);
    }
}

static void vTarea_Boton(void *pvParams) {
    for (;;) {
        xSemaphoreTake(xSemBtn, portMAX_DELAY);

        if (!armando) {
            /* Inicia el proceso de armado */
            armando = 1;
            HAL_GPIO_WritePin(LED2_PORT, LED2_PIN, GPIO_PIN_RESET);
            xTimerChangePeriod(xTimer_Blink, pdMS_TO_TICKS(100), 0);
            xTimerStart(xTimer_Blink, 0);
            xTimerStart(xTimer_Arm,   0);
        } else {
            /* Cancela el armado */
            xTimerStop(xTimer_Blink, 0);
            xTimerStop(xTimer_Arm,   0);
            HAL_GPIO_WritePin(LED1_PORT, LED1_PIN, GPIO_PIN_RESET);
            HAL_GPIO_WritePin(LED2_PORT, LED2_PIN, GPIO_PIN_RESET);
            armando = 0;
        }

        vTaskDelay(pdMS_TO_TICKS(300));   /* Anti-rebote */
    }
}

int main(void) {
    HAL_Init();
    SystemClock_Config();
    MX_GPIO_Init();

    xSemBtn      = xSemaphoreCreateBinary();
    xTimer_Blink = xTimerCreate("BLINK", pdMS_TO_TICKS(100),   pdTRUE,  NULL, vCB_Blink);
    xTimer_Arm   = xTimerCreate("ARM",   pdMS_TO_TICKS(10000), pdFALSE, NULL, vCB_Armado);

    xTaskCreate(vTarea_Boton, "BTN", 128, NULL, 1, NULL);

    vTaskStartScheduler();
    for (;;);
}
```

### Análisis — Desafío 7

- **¿Cómo se comunican los timers entre sí?** A través de handles globales y la bandera `armando`. El callback `vCB_Armado` llama `xTimerStop(xTimer_Blink, 0)` usando el handle del otro timer.
- **¿Es seguro modificar o detener un timer desde el callback de otro timer?** **Sí.** `xTimerStop`, `xTimerStart` y `xTimerChangePeriod` no modifican directamente al timer: envían un **comando** a la cola interna del Timer Daemon. El Timer Daemon procesa esa cola de forma serializada, por lo que no hay condición de carrera.
- **Variable `armando`:** debe ser `volatile` para evitar que el compilador la optimice en registros, ya que se accede desde múltiples contextos (tarea y callback del Timer Daemon).